# Assignment 4
### Do three of four.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

### Exercise 1: Contingent Comparisons
- Load the Minnesota use of force data.
- Bootstrap the proportion of missing values for `subject_injury` for each race, and plot the results with grouped KDE and ECDF plots
- Describe what you see. When we consider second order uncertainty, how similar or different are the sampling distributions of these proportions? 

In [ ]:
df=pd.read_csv('/Users/graceegeorge/DS5030_public_repo/data/mn_police_use_of_force.csv')
df.head()

In [ ]:
df['injury_na']=df['subject_injury'].isna()
# df

In [ ]:
pd.crosstab(df['injury_na'],df['race'],normalize='columns')

In [ ]:
race_injury=df.loc[:,['race','injury_na']].groupby('race').value_counts()
race_injury

In [ ]:
estimates=[]
for i in range(2500):
    x=df.sample(frac=1,replace=True)
    estimate=x.loc[:,['race','injury_na']].groupby('race').mean()
    estimates.append(estimate)
new_df=pd.concat([row for row in estimates],axis=1).T
new_df
    




In [ ]:
# Used AI for help debugging
sns.ecdfplot(new_df)
plt.show()

In [ ]:
sns.kdeplot(new_df)
plt.show()

The sampling distrubution of the other/mixed race category and the Asian category seem to have lower proportions of na injury values compared to the other races. The distributions of Native Americans ,white people, and black people  look pretty similar and overlap. I found it intersting that the ecdf for the pacific islander group is a straight line a 100%. I wonder if this has to do with there being less pacific islanders in the population, so there will be less in the sample.

### Exercise 2: Invitation to Inference
- Run the simulation code line by line and comment what each line is doing, or write your own code to do the resampling
- Open the NHANES or Ames prices or College Completion data. Pick a variable and a statistic to compute (e.g. mean, median, variance, IQR)
- Use the `simulate` function from class to get a sample of estimates for your statistic and your data
- Create a new function, `interval(L,H,estimates)`, that computes the $L$-th and $H$-th quantiles for your estimates, $H>L$
- If $L=.05$ and $H=.95$, this is a **90-percent confidence interval**: "For our statistic, this interval captures the true value of the population parameter 90 percent of the time. (We are 90% **confident** that it includes the true value of the parameter, but the probability that the true parameter lies in this interval is 0 or 1.)"
- We will spend much more time on this later in class, but for people who have done hypothesis testing before, you now know how to do it directly from the data: No central limit theorem required.

In [ ]:
df=pd.read_csv('/Users/graceegeorge/DS5030_public_repo/data/ames_prices.csv')
# df.head()

In [ ]:
def simulate(var, df, fcn, S=1000, plot=True): # Taking a sample the df data
                                                # applying the fcn function S times and 
                                                # makes plot of results
    ''' Bootstrap simulation code. '''    
    x = df[var]     #Sets the variable being bootstrapped

    estimates = []     # Sets an place to store bootrapped estimates
    for s in range(S): # Takes S samples
        x_s = x.sample(frac=1.0,replace=True) # Takes a sample of x's while replacing each time
        stat_s = fcn(x_s) #Takes the desired stat of each variable
        estimates.append(stat_s) # Stores each estimated stat in a list
    estimates = np.array(estimates) # Convert estimates list to numpy array

    ## or in one line, 

    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))  # sets dimension of figures
        sns.kdeplot(x, ax = axes[0]).set(title='KDE of Underlying Data') # Makes a kde of the data(not sampled)
        sns.kdeplot(estimates, ax = axes[1]).set(title='KDE of Computed Statistics') # Makes a kde of the bootstapimg
        axes[1].axvline(x=fcn(x), color='orange', linestyle='--') # Data visualization
        sns.ecdfplot(estimates, ax = axes[2]).set(title='ECDF of Computed Statistics') # Makes an ecdf of the bootrapped stats
        axes[2].axvline(x=fcn(x), color='orange', linestyle='--') # Data visualization
        plt.show() # Prints the graphs
        print(f'Variance of estimates is: {np.var(estimates)}') # Prints the variance of the estimates
        
    return estimates # Returns the list that stored the estimates

In [ ]:
def stat(x):
    stat=np.mean(x)
    return stat

In [ ]:
estimates=simulate('area',df,stat)


In [ ]:
def interval(L,H,estimates,plot=True):
    if L<H:
        Lth=np.quantile(estimates,L)
        print(Lth)
        print(Lth)
        Hth=np.quantile(estimates,H)
        print()
    else:
        return   
    if plot:
        sns.kdeplot(estimates)
        plt.axvline(x=Lth,color='orange',linestyle="--") 
        plt.axvline(x=Hth,color='orange',linestyle="--") 
        print(f'Lower Bound:{Lth} \n Upper Bound:{Hth}')
    return Lth,Hth
        

In [ ]:
interval(.05,.95,estimates,plot=True)

We are 95% confident that the true value of average house area lies between 1485 and 1515.

### Exercise 3: Intro to A/B Testing
- Go here, and read about this study: https://www.clinicaltrials.gov/study/NCT01985360
- Read the Study Overview and explain what the goal of the trial is 
- Read the Study Plan and explain how it was designed and why -- there's lots of medical jargon, but the main point is how patients were assigned to interventions. 
- Read the Results Posted: Go to **Outcome Measures**. Explain how table 1 ("Incidence of Death from Any Cause or Myocardial Infarction") is a contingency table. These are the data for this exercise.
- What is the difference in surival rates between the invasive strategy and the conservative strategy?
- Bootstrap the survival rates for the two groups, and plot them as KDEs and ECDFs against one another
- Bootstrap the difference in surival rates, and plot it as a KDE and ECDF
- Is this an effective health intervention? Explain your answer clearly

This would be what CS people call **A/B testing** and everyone else called a **randomized controlled trial**: Using randomized assignment to detect the difference in outcomes between two groups. (We've just done a non-parametric version of a two-sample t-test.)

Used answers for guidance.

The goal of the trial is to determine the best managment stategy for patients with stable ischemic heart disease (SIHD). The primary aim was to see if an invasive strategy or conservative will be better at reducing the primary composite of death of nonfatal myocardial infraction.

Patients were randomly assinged to interventions.

Table 1 is a contigency table because it shows the values of two variables number of particpant analyzed and measure type for both treatment groups.

The two sided difference in survival rates is between 0.79 and 1.29

In [ ]:
n_con = 389
y_con = np.ones(n_con)
y_con[:129] = 0
df_con = pd.DataFrame({'arm':'conservative','outcome':y_con})



In [ ]:
n_con=388
y_inv=np.ones(n_con)
y_inv[:129]=0
df_inv = pd.DataFrame({'arm':'invasive','outcome':y_inv})
df_inv.head


In [ ]:
df=pd.concat([df_con,df_inv])
df

In [ ]:
est_con=[]
est_inv=[]
est_diff=[]
for i in range(100):
    con=df_con.sample(frac=1.0,replace=True)
    inv=df_inv.sample(frac=1.0,replace=True)
    est_con.append(con['outcome'].mean())
    est_inv.append(inv['outcome'].mean())
    est_diff.append(con['outcome'].mean()-inv['outcome'].mean())
print(est_con)
print(est_inv)
print(est_diff)
    


In [ ]:
sns.kdeplot(est_con,label="Conservative")
sns.kdeplot(est_inv,label='Invasive')
plt.legend()
plt.show()

In [ ]:
sns.ecdfplot(est_con,label="Conservative")
sns.ecdfplot(est_inv,label='Invasive')
plt.legend()
plt.show()

In [ ]:
sns.kdeplot(est_diff)
plt.show()
sns.kdeplot(est_diff)
plt.show()

It doesn't seem like there is a difference between the two treatments. Looking at the difference kde and ecdf plots, it seems that 0 is the mean difference, suggesting there is no difference.

### Exercise 4: Prediction Uncertainty
- Pick a dataset and two continuous variables.
- Recall the LCLS estimator:
$$
\hat{y}(z) =  \frac{ \frac{1}{N} \sum_{i=1}^N y_i \times \frac{1}{h}k\left( \frac{z - x_i}{h} \right)}{ \frac{1}{N} \sum_{i=1}^N \frac{1}{h} k\left( \frac{z - x_i}{h} \right)}
$$
with the Epanechnikov kernel and the standard plug-in bandwidth for $h$
- Compute and plot this line for 30 bootstrap samples. Notice where there is a lot of variation in the predictions, versus little variation in the predictions.
- Now, for any $z$, we can bootstrap a distribution of predictions using the above formula. Do this at the 25th percentile, median, and 75th percentile of $X$, and make KDE plots of your results.
- Now, pick a grid for $z$: Obvious choices are all of the unique values in the data, or an equally spaced grid from the minimum value to the maximum value. For each $z$, bootstrap a sample of predictions and compute the .05 and .95 quantiles. Plot these error curves along with your LCLS estimate. Where are your predictions "tight"/reliable? Where are they highly variable/unreliable?